# rawfilereader-cli — Interactive Testing Notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mzzzhunter/rawfilereader-cli/blob/claude/cli-file-access-agent-xsdMh/notebooks/rawfilereader_cli_colab.ipynb)

This notebook walks through every command exposed by **rawfilereader-cli** against a real Thermo Fisher `.raw` file.

**Sections**
1. [Setup](#setup) — install .NET, the CLI, and download a sample file
2. [file commands](#file) — metadata, chromatograms, peak detection
3. [scan commands](#scan) — per-scan spectra, stats, trailer data
4. [search commands](#search) — lookup by filter string or retention time
5. [analyze commands](#analyze) — summaries, averaging, scan iteration

---
## 1. Setup <a id="setup"></a>

In [ ]:
# Install the .NET 8 runtime (required by the Thermo RawFileReader .NET library)
%%bash
set -e
wget -q https://dot.net/v1/dotnet-install.sh -O /tmp/dotnet-install.sh
bash /tmp/dotnet-install.sh --channel 8.0 --runtime dotnet 2>&1 | tail -3
echo "Done"

In [ ]:
import os

home = os.path.expanduser("~")
dotnet_root = os.path.join(home, ".dotnet")
os.environ["DOTNET_ROOT"] = dotnet_root
os.environ["PATH"] = dotnet_root + ":" + dotnet_root + "/tools:" + os.environ.get("PATH", "")
print("DOTNET_ROOT:", dotnet_root)

In [ ]:
# Install the Thermo adapter (with all dependencies such as pythonnet)
!pip install -q "git+https://github.com/mzzzhunter/RawFileReaderPyAdapter.git"
# Force-reinstall rawfilereader-cli from the pinned branch.
# --no-deps avoids pulling a stale cached adapter on repeated runs.
!pip install -q --force-reinstall --no-deps "git+https://github.com/mzzzhunter/rawfilereader-cli.git@claude/cli-file-access-agent-xsdMh"

In [ ]:
import os, subprocess, pathlib

libs_dir = pathlib.Path("/tmp/rawfilereader_adapter/libs/Net8/Assemblies")

if not libs_dir.exists():
    print("Downloading RawFileReader assemblies (sparse clone) …")
    subprocess.run(
        ["git", "clone", "--depth", "1", "--filter=blob:none", "--sparse",
         "https://github.com/mzzzhunter/RawFileReaderPyAdapter.git",
         "/tmp/rawfilereader_adapter"],
        check=True, capture_output=True,
    )
    subprocess.run(
        ["git", "sparse-checkout", "set", "libs/Net8/Assemblies"],
        cwd="/tmp/rawfilereader_adapter", check=True, capture_output=True,
    )

os.environ["RAWFILEREADER_LIBS"] = str(libs_dir)
dlls = sorted(libs_dir.glob("*.dll"))
print(f"RAWFILEREADER_LIBS = {libs_dir}")
print(f"Found {len(dlls)} DLL(s): {[d.name for d in dlls]}")

In [ ]:
import urllib.request

RAW_FILE = "sample.raw"
url = "https://github.com/mzzzhunter/RawFileReaderPyAdapter/raw/refs/heads/main/sample.raw"

if not os.path.exists(RAW_FILE):
    print("Downloading sample.raw …")
    urllib.request.urlretrieve(url, RAW_FILE)

print(f"sample.raw  {os.path.getsize(RAW_FILE) / 1e6:.1f} MB")

### Use your own `.raw` file (optional)

Run the cell below **instead of** (or after) the sample-download cell above to analyse your own Thermo Fisher `.raw` file.  
`RAW_FILE` will be updated automatically and every subsequent cell will use it.

In [ ]:
# ── Upload your own .raw file ──────────────────────────────────────────────
# Run this cell to use your own Thermo Fisher .raw file instead of sample.raw.
# Skip it (or don't run it) to keep using the downloaded sample.
from google.colab import files as _colab_files

_uploaded = _colab_files.upload()          # opens the file picker
if _uploaded:
    RAW_FILE = next(iter(_uploaded))       # use the first uploaded filename
    print(f"RAW_FILE set to: {RAW_FILE!r}  ({os.path.getsize(RAW_FILE) / 1e6:.1f} MB)")
else:
    print("No file uploaded — keeping current RAW_FILE:", RAW_FILE)

In [ ]:
import subprocess
import json
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
%matplotlib inline
plt.rcParams["figure.dpi"] = 110

def run_cli(*args, stream=False):
    """Run a rawfilereader command and return parsed JSON.

    Pass CLI tokens as positional args (e.g. 'file', 'info').
    --file is appended automatically.
    Set stream=True to get a list of objects from newline-delimited JSON.
    """
    cmd = ["rawfilereader", "--indent", "2"] + list(args) + ["--file", RAW_FILE]
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(r.stderr.strip())
    if stream:
        return [json.loads(line) for line in r.stdout.splitlines() if line.strip()]
    return json.loads(r.stdout)

print("Helper ready. RAW_FILE =", RAW_FILE)
print("Run 'rawfilereader --help' for the full command tree.")

---
## 2. file commands <a id="file"></a>

| Command | Description |
|---|---|
| `file info` | File metadata and run header |
| `file scan_range` | First and last scan number |
| `file instrument` | Instrument count and metadata |
| `file filters` | All unique scan filter strings |
| `file chromatogram` | Extract a chromatogram trace |
| `file chromatogram_peaks` | Detect peaks in a chromatogram |

In [ ]:
# ── file info ──────────────────────────────────────────────────────────────
info = run_cli("file", "info")
print("File name  :", info["file_info"].get("name", "—"))
print("Created    :", info["file_info"].get("creation_date", "—"))
print("Operator   :", info["file_info"].get("operator", "—"))
print("Time range :", info["run_header"].get("time_range", "—"), "min")

In [ ]:
# ── file scan_range ────────────────────────────────────────────────────────
sr = run_cli("file", "scan_range")
first_scan = sr["first_scan"]
last_scan  = sr["last_scan"]
print(f"Scans: {first_scan} – {last_scan}  (total {last_scan - first_scan + 1})")

In [ ]:
# ── file instrument ────────────────────────────────────────────────────────
inst = run_cli("file", "instrument")
print("Instrument count:", inst["count"])
ii = inst["instrument_info"]
for key in ("name", "model", "serial_number", "device_type"):
    if key in ii:
        print(f"  {key:15s}: {ii[key]}")

In [ ]:
# ── file chromatogram — BasePeak, TIC & mass-range SIC ────────────────────
bp  = run_cli("file", "chromatogram", "--trace_type", "BasePeak")
tic = run_cli("file", "chromatogram", "--trace_type", "TIC")
sic = run_cli("file", "chromatogram", "--trace_type", "mass_range", "--mass_range", "200.0-500.0")

fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(10, 7), sharex=True)

ax1.plot(bp["times"], bp["intensities"], lw=0.8, color="steelblue")
ax1.set_ylabel("Intensity")
ax1.set_title("Base Peak Chromatogram")
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1e6:.1f}M"))

ax2.plot(tic["times"], tic["intensities"], lw=0.8, color="darkorange")
ax2.set_ylabel("Intensity")
ax2.set_title("Total Ion Chromatogram (TIC)")
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1e6:.1f}M"))

ax3.plot(sic["times"], sic["intensities"], lw=0.8, color="seagreen")
ax3.set_ylabel("Intensity")
ax3.set_xlabel("Retention time (min)")
ax3.set_title("Selected Ion Chromatogram (SIC) — m/z 200–500")
ax3.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1e6:.1f}M"))

plt.tight_layout()
plt.show()

In [ ]:
# ── file chromatogram — BasePeak & TIC ─────────────────────────────────────
bp  = run_cli("file", "chromatogram", "--trace_type", "BasePeak")
tic = run_cli("file", "chromatogram", "--trace_type", "TIC")

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 5), sharex=True)

ax1.plot(bp["times"], bp["intensities"], lw=0.8, color="steelblue")
ax1.set_ylabel("Intensity")
ax1.set_title("Base Peak Chromatogram")
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1e6:.1f}M"))

ax2.plot(tic["times"], tic["intensities"], lw=0.8, color="darkorange")
ax2.set_ylabel("Intensity")
ax2.set_xlabel("Retention time (min)")
ax2.set_title("Total Ion Chromatogram (TIC)")
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1e6:.1f}M"))

plt.tight_layout()
plt.show()

In [ ]:
# ── file chromatogram_peaks ────────────────────────────────────────────────
# Adjust MIN_HEIGHT to suppress noise peaks.  Set to 0 to keep all local maxima.
MIN_HEIGHT = 1e5   # e.g. 1e5 = 100 000,  1e6 = 1 000 000

peaks_data = run_cli(
    "file", "chromatogram_peaks",
    "--trace_type",    "BasePeak",
    "--smooth_window", "7",
    "--min_height",    str(MIN_HEIGHT),
)

print(f"Detected {peaks_data['peak_count']} peaks  "
      f"(smooth_window={peaks_data['smooth_window']}, "
      f"min_height={peaks_data['min_height']:.2e})")
print("\nTop 5 peaks by intensity:")
for p in peaks_data["peaks"][:5]:
    print(f"  RT {p['retention_time']:6.3f} min   intensity {p['intensity']:.3e}   smoothed {p['smoothed_intensity']:.3e}")

# Plot smoothed chromatogram with peak markers
peak_rts  = [p["retention_time"]    for p in peaks_data["peaks"]]
peak_ints = [p["smoothed_intensity"] for p in peaks_data["peaks"]]

plt.figure(figsize=(10, 3.5))
plt.plot(peaks_data["times"], peaks_data["smoothed_intensities"],
         lw=0.9, color="steelblue",
         label=f"BasePeak (smoothed w={peaks_data['smooth_window']})")
plt.scatter(peak_rts, peak_ints, color="red", s=18, zorder=5,
            label=f"{len(peak_rts)} peaks ≥ {MIN_HEIGHT:.0e}")
plt.xlabel("Retention time (min)")
plt.ylabel("Intensity")
plt.title(f"Detected chromatogram peaks  (min_height={MIN_HEIGHT:.0e})")
plt.gca().yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1e6:.1f}M"))
plt.legend()
plt.tight_layout()
plt.show()

---
## 3. scan commands <a id="scan"></a>

| Command | Description |
|---|---|
| `scan info` | Scan metadata (MS order, RT, filter …) |
| `scan stats` | TIC, base peak mass/intensity |
| `scan spectrum` | Centroid m/z + intensity arrays |
| `scan profile` | Profile (continuous) spectrum |
| `scan filter` | Thermo filter string for one scan |
| `scan trailer` | Trailer / auxiliary values |
| `scan dependents` | Parent–dependent scan hierarchy |

In [ ]:
# ── scan info — single scan ────────────────────────────────────────────────
si = run_cli("scan", "info", "--scan_number", str(first_scan))
print(f"Scan #{si['scan_number']}")
for key in ("ms_order", "retention_time", "filter", "centroid_flag", "detector_type"):
    if key in si:
        print(f"  {key:20s}: {si[key]}")

In [ ]:
# ── scan info — iterate all MS1 scans (returned as JSON array) ─────────────
ms1_scans = run_cli("scan", "info", "--ms_order", "1")
print(f"Total MS1 scans: {len(ms1_scans)}")
print("First MS1 scan:", ms1_scans[0]["scan_number"],
      "  RT:", ms1_scans[0].get("retention_time"), "min")
# Store a couple of MS1 scan numbers for later
ms1_scan = ms1_scans[0]["scan_number"]

In [ ]:
# ── scan stats ─────────────────────────────────────────────────────────────
stats = run_cli("scan", "stats", "--scan_number", str(ms1_scan))
print(f"Scan #{ms1_scan} statistics")
for key in ("tic", "base_peak_mass", "base_peak_intensity", "mass_range", "centroid_flag"):
    if key in stats:
        print(f"  {key:25s}: {stats[key]}")

In [ ]:
# ── scan spectrum ──────────────────────────────────────────────────────────
spec = run_cli("scan", "spectrum", "--scan_number", str(ms1_scan))
masses = spec["masses"]
intens = spec["intensities"]
print(f"Scan #{ms1_scan}  —  {spec['point_count']} centroids")

plt.figure(figsize=(10, 3.5))
plt.vlines(masses, 0, intens, lw=0.4, color="steelblue")
plt.xlabel("m/z")
plt.ylabel("Intensity")
plt.title(f"Centroid spectrum — scan {ms1_scan}")
plt.gca().yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1e6:.1f}M"))
plt.tight_layout()
plt.show()

In [ ]:
# ── scan spectrum — truncate arrays with --max_points ──────────────────────
spec_top = run_cli("scan", "spectrum", "--scan_number", str(ms1_scan), "--max_points", "20")
print(f"Returned {len(spec_top['masses'])} of {spec_top['point_count']} points  (truncated={spec_top['truncated']})")

# --max_points -1  returns only metadata (no arrays)
meta_only = run_cli("scan", "spectrum", "--scan_number", str(ms1_scan), "--max_points", "-1")
print(f"\nMetadata-only mode: masses={meta_only['masses']}  point_count={meta_only['point_count']}")

In [ ]:
# ── scan profile ───────────────────────────────────────────────────────────
prof = run_cli("scan", "profile", "--scan_number", str(ms1_scan))
mz          = prof["segments"][0][0]
intensities = prof["segments"][0][1]

print("Profile data structure:")
for k, v in prof.items():
    if isinstance(v, list):
        print(f"  '{k}': list[{len(v)}]  sample={v[:3]}")
    else:
        print(f"  '{k}': {v!r}")
print(f"\nm/z range : {mz[0]:.2f} – {mz[-1]:.2f}")
print(f"Points    : {len(mz)}")

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(mz, intensities, lw=0.6, color="darkorange")
ax.fill_between(mz, intensities, alpha=0.15, color="darkorange")
ax.set_xlabel("m/z")
ax.set_ylabel("Intensity")
ax.set_title(f"Profile spectrum — scan {ms1_scan}")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1e6:.1f}M"))
ax.set_xlim(mz[0], mz[-1])
ax.grid(axis="y", lw=0.4, alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
# ── scan filter ────────────────────────────────────────────────────────────
flt_scan = run_cli("scan", "filter", "--scan_number", str(ms1_scan))
print(f"Scan {flt_scan['scan_number']} filter:\n  {flt_scan['filter']}")

In [ ]:
# ── scan trailer ───────────────────────────────────────────────────────────
trailer = run_cli("scan", "trailer", "--scan_number", str(ms1_scan))
print(f"Trailer data for scan {ms1_scan}:")
fields = trailer.get("fields", trailer)  # shape depends on adapter version
if isinstance(fields, dict):
    for k, v in list(fields.items())[:10]:
        print(f"  {k:35s}: {v}")
else:
    print(trailer)

In [ ]:
# ── scan dependents ────────────────────────────────────────────────────────
# Find an MS2 scan to illustrate parent/dependent relationships
try:
    ms2_scans = run_cli("scan", "info", "--ms_order", "2")
    if ms2_scans:
        ms2_scan = ms2_scans[0]["scan_number"]
        # Walk up to the parent MS1 scan
        dep = run_cli("scan", "dependents", "--scan_number", str(ms2_scan - 1), "--depth", "1")
        print(f"Dependents of scan {ms2_scan - 1}:")
        print(json.dumps(dep, indent=2))
    else:
        print("No MS2 scans found — skipping dependents demo")
except Exception as e:
    print("Note:", e)

---
## 4. search commands <a id="search"></a>

| Command | Description |
|---|---|
| `search by_filter` | All scan numbers matching a filter string |
| `search by_rt` | Scan number closest to an RT (minutes) |
| `search rt_for_scan` | RT (minutes) for a given scan number |
| `search iterate_filter` | Scan numbers in an optional RT window |

In [ ]:
# ── search by_filter ───────────────────────────────────────────────────────
result = run_cli("search", "by_filter", "--filter_string", ms1_filter)
print(f"Filter: {result['filter_string']!r}")
print(f"Matching scans: {result['count']}  (first 10: {result['scan_numbers'][:10]})")

In [ ]:
# ── search by_rt ───────────────────────────────────────────────────────────
# Derive actual RT range from the file (run_header.time_range is not always present)
rt_first = run_cli("search", "rt_for_scan", "--scan_number", str(first_scan))["retention_time"]
rt_last  = run_cli("search", "rt_for_scan", "--scan_number", str(last_scan))["retention_time"]
run_time = [rt_first, rt_last]   # used by iterate_filter cell below
mid_rt_val = round((rt_first + rt_last) / 2, 3)
print(f"File RT range: {rt_first:.4f} – {rt_last:.4f} min  →  querying midpoint {mid_rt_val} min")

by_rt = run_cli("search", "by_rt", "--retention_time", str(mid_rt_val))
print(f"RT {by_rt['retention_time']} min  →  scan #{by_rt['scan_number']}")

In [ ]:
# ── search by_rt ───────────────────────────────────────────────────────────
run_time = info["run_header"].get("time_range", [0, 10])
mid_rt_val = round((run_time[0] + run_time[1]) / 2, 3)

by_rt = run_cli("search", "by_rt", "--retention_time", str(mid_rt_val))
print(f"RT {by_rt['retention_time']} min  →  scan #{by_rt['scan_number']}")

In [ ]:
# ── search rt_for_scan ─────────────────────────────────────────────────────
rt_result = run_cli("search", "rt_for_scan", "--scan_number", str(ms1_scan))
print(f"Scan #{rt_result['scan_number']}  →  RT {rt_result['retention_time']:.4f} min")

In [ ]:
# ── search iterate_filter — array mode ────────────────────────────────────
iter_result = run_cli(
    "search", "iterate_filter",
    "--filter_string", ms1_filter,
    "--start_time", str(run_time[0]),
    "--end_time",   str(run_time[1]),
)
print(f"iterate_filter returned {iter_result['count']} scans")
print("First 5:", iter_result["scan_numbers"][:5])

In [ ]:
# ── search iterate_filter — stream (NDJSON) mode ──────────────────────────
# Remove --indent; stream flag makes rawfilereader emit one JSON object per line
cmd = [
    "rawfilereader", "search", "iterate_filter",
    "--filter_string", ms1_filter,
    "--stream",
    "--file", RAW_FILE,
]
r = subprocess.run(cmd, capture_output=True, text=True)
stream_objs = [json.loads(line) for line in r.stdout.splitlines() if line.strip()]
print(f"Stream mode: {len(stream_objs)} objects")
print("First 3 objects:", stream_objs[:3])

---
## 5. analyze commands <a id="analyze"></a>

| Command | Description |
|---|---|
| `analyze summary` | Scan count by MS order |
| `analyze average_scans` | Average spectra across a scan range |
| `analyze scan_info_range` | Iterate scan metadata (optionally filtered by MS order) |

In [ ]:
# ── analyze summary ────────────────────────────────────────────────────────
summ = run_cli("analyze", "summary")
print("Scan counts by MS order:")
for ms_order, count in sorted(summ["summary"].items()):
    print(f"  {ms_order:5s}: {count}")

# Pie chart
labels = list(summ["summary"].keys())
values = list(summ["summary"].values())

fig, ax = plt.subplots(figsize=(7, 6))
wedges, texts, autotexts = ax.pie(
    values,
    labels=None,
    autopct="%1.1f%%",
    startangle=90,
    pctdistance=0.75,
)
for t in autotexts:
    t.set_fontsize(11)
ax.legend(wedges, labels, title="MS order", loc="center left",
          bbox_to_anchor=(1, 0, 0.5, 1), fontsize=10)
ax.set_title("Scan distribution by MS order", fontsize=13, pad=14)
plt.tight_layout()
plt.show()

In [ ]:
# ── analyze average_scans ──────────────────────────────────────────────────
n_avg = min(20, last_scan - first_scan)  # average up to 20 scans
avg = run_cli(
    "analyze", "average_scans",
    "--first_scan", str(first_scan),
    "--last_scan",  str(first_scan + n_avg),
    "--filter_string", ms1_filter,
)
print(f"Averaged scans {avg['first_scan']}–{avg['last_scan']}  ({avg['point_count']} data points)")

plt.figure(figsize=(10, 3.5))
plt.vlines(avg["masses"], 0, avg["intensities"], lw=0.4, color="seagreen")
plt.xlabel("m/z")
plt.ylabel("Intensity")
plt.title(f"Averaged spectrum — scans {avg['first_scan']}–{avg['last_scan']}")
plt.gca().yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1e6:.1f}M"))
plt.tight_layout()
plt.show()

In [ ]:
# ── analyze scan_info_range — MS1 only, stream mode ───────────────────────
cmd = [
    "rawfilereader", "analyze", "scan_info_range",
    "--ms_order", "1",
    "--stream",
    "--file", RAW_FILE,
]
r = subprocess.run(cmd, capture_output=True, text=True)
ms1_info = [json.loads(line) for line in r.stdout.splitlines() if line.strip()]
print(f"Streamed {len(ms1_info)} MS1 scan info objects")

# Plot injection time vs RT
rts      = [s.get("retention_time") for s in ms1_info if s.get("injection_time") is not None]
inj_time = [s.get("injection_time")  for s in ms1_info if s.get("injection_time") is not None]

if rts:
    plt.figure(figsize=(9, 3))
    plt.plot(rts, inj_time, lw=0.7, color="purple")
    plt.xlabel("Retention time (min)")
    plt.ylabel("Injection time (ms)")
    plt.title("MS1 ion injection time across the run")
    plt.tight_layout()
    plt.show()
else:
    print("injection_time not available in scan_info for this file")

---
## JSON output format

Every command writes JSON to **stdout**. Errors write to **stderr** and exit 1:

```json
// stderr on error
{"error": "Scan 9999 not found", "type": "scan_not_found", "details": {}}
```

Error types: `raw_file_error`, `not_open_error`, `scan_not_found`, `instrument_error`, `assembly_load_error`, `in_acquisition_error`, `unexpected_error`.

In [ ]:
# ── Error handling demo ────────────────────────────────────────────────────
r = subprocess.run(
    ["rawfilereader", "scan", "spectrum", "--scan_number", "999999", "--file", RAW_FILE],
    capture_output=True, text=True,
)
print("Exit code:", r.returncode)
if r.stderr.strip():
    err = json.loads(r.stderr)
    print("Error type :", err["type"])
    print("Error msg  :", err["error"])